# Explorative Datenanalyse
Erste Sichtung der Kursdaten fuer die acht Portfolio-Titel: Kursverlaeufe,
Renditen, Verteilungen, Korrelationen und Sektorzuordnung. Grundlage
fuer die Methodik des Berichts und Sanity-Check der heruntergeladenen
Daten.

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from portfolio_oracle.config import load_config
from portfolio_oracle.data.loader import load_all_tickers, load_close_panel

sns.set_theme(style="whitegrid")
config = load_config()
tickers = config["portfolio"]["tickers"]
print("Tickers:", tickers)

## Kursverlaeufe (bereinigte Close-Kurse)

In [ ]:
closes = load_close_panel()
closes.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
closes.plot(ax=ax)
ax.set_title("Bereinigte Schlusskurse 2017-2026")
ax.set_ylabel("Kurs in USD")
ax.set_xlabel("Datum")
plt.tight_layout()
plt.show()

## Normalisierte Verlaeufe (Start = 100)
Damit Titel mit unterschiedlichem Kursniveau vergleichbar werden.

In [ ]:
normed = closes.div(closes.iloc[0]).mul(100)
fig, ax = plt.subplots(figsize=(12, 6))
normed.plot(ax=ax)
ax.set_title("Normalisierte Kursverlaeufe (Start = 100)")
ax.set_ylabel("Indexwert")
plt.tight_layout()
plt.show()

## Tagesrenditen und Verteilungen

In [ ]:
returns = closes.pct_change().dropna()
returns.describe()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for ax, ticker in zip(axes.flat, returns.columns):
    sns.histplot(returns[ticker], bins=60, ax=ax)
    ax.set_title(ticker)
    ax.set_xlabel("Tagesrendite")
plt.tight_layout()
plt.show()

## Korrelationsmatrix der Tagesrenditen

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Korrelation der Tagesrenditen")
plt.tight_layout()
plt.show()

## Sektorzuordnung
Acht Titel ueber sechs Sektoren gemaess Exposee.

In [ ]:
sectors = {
    "AAPL": "Technologie",
    "NVDA": "Technologie",
    "AMZN": "Konsumgueter",
    "PG": "Konsumgueter",
    "UPS": "Industrie",
    "XOM": "Energie",
    "JPM": "Finanzen",
    "JNJ": "Pharma",
}
sector_df = pd.DataFrame(
    {"Ticker": list(sectors.keys()), "Sektor": list(sectors.values())}
)
sector_df.sort_values("Sektor")

## Fazit
- Alle acht Titel decken den Zeitraum 2017 bis 2026 ab.
- Die normalisierten Verlaeufe zeigen deutliche Unterschiede zwischen
  Wachstumstiteln (NVDA, AAPL, AMZN) und defensiven Werten (PG, JNJ).
- Die Korrelationen liegen im erwartbaren Bereich; das Portfolio ist
  ueber sechs Sektoren diversifiziert.